# Train NNUE with king buckets, pawn structure, piece counts and mobility (Kaggle)

This notebook does everything by itself with **Save & Run All** (GPU and Internet on):

1. Quick check of the whole pipeline on a small sample (fails within minutes if something is wrong)
2. Streams **all** quiet, deeply evaluated positions from the Lichess evaluation database (416 million positions in the file; only positions with depth 25 or more and a quiet best move are kept)
3. Lets Optuna choose the number of king buckets, the network size and the learning rate, within a speed and file size limit
4. Trains the best network for as many epochs as fit in the time budget
5. Writes `nnue.bin` (format NNU3), ready for the Rust engine, and prints the piece values the network learned

Inputs per perspective:
- 768 piece-square inputs per king bucket (king position, horizontally mirrored)
- 384 pawn structure inputs: passed, isolated and doubled pawns, own and enemy
- 100 piece count inputs: how many pawns, knights, bishops, rooks and queens each side has (0 to 9 or more)
- 640 mobility inputs: for every knight, bishop, rook, queen and king, how many free squares it has in each direction before it is blocked; for the king all 8 queen directions are used, so these inputs also show how open the king is
- During training only: 768 piece-square inputs shared by all buckets and 12 piece count inputs; these are added into the bucket weights when exporting

In [ ]:
!pip install -q optuna zstandard

In [ ]:
import collections
import json
import math
import multiprocessing
import os
import struct
import time
import urllib.request

import numpy as np
import optuna
import torch
import zstandard
from matplotlib import pyplot as plt
from torch import nn

## Settings

In [ ]:
notebook_start = time.time()

max_positions = 420_000_000
max_load_minutes = 240
min_depth = 25
val_size = 2_000_000
test_size = 2_000_000
batch_size = 4096
gpu_data_fraction = 0.6

bucket_options = [1, 4, 8, 16]
hidden_options = [128, 256, 384, 512]
layer2_options = [0, 8, 16]
max_cost = 20_480
max_size_mb = 32

num_trials = 30
tune_hours = 2.0
tune_epochs = 3
tune_train_size = 10_000_000
tune_val_size = 1_000_000

max_epochs = 30
total_hours = 11.5
reserve_minutes = 30

CP_SCALE = 0.00184104
PIECE_VALUES = [100, 300, 320, 500, 900, 0]

url = "https://database.lichess.org/lichess_db_eval.jsonl.zst"
out_dir = "/kaggle/working"
log_file = os.path.join(out_dir, "nnue_mobility.log")
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
num_workers = os.cpu_count()

print("device:", device, "| CPU cores:", num_workers)

## Helpers: cost and file size of each architecture

The cost is the number of weight additions per move in the engine. Measured on 48,000 positions from the engine's own games: a move changes about 14 inputs per perspective (about 3 without counts and mobility), and a king bucket change refreshes about 89 inputs. `max_cost` allows at most twice the work of v11 (1024 hidden, no buckets: 10,240).

In [ ]:
def log(text):
    hours = (time.time() - notebook_start) / 3600
    line = f"[{time.strftime('%H:%M:%S')} | {hours:.2f} h] {text}"
    print(line, flush=True)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize()


def compute_cost(num_hidden, num_layer2):
    update = 28 * num_hidden
    refresh = 12 * num_hidden
    if num_layer2 == 0:
        return update + refresh + 2 * num_hidden
    return update + refresh + 2 * num_hidden * num_layer2 + num_layer2


def file_size_mb(num_buckets, num_hidden):
    return (num_buckets * 768 + 384 + 100 + 640) * (num_hidden + 1) * 4 / 1e6


def parse_architecture(architecture):
    return tuple(int(part) for part in architecture.split("x"))


architectures = [
    f"{b}x{h}x{l}"
    for b in bucket_options for h in hidden_options for l in layer2_options
    if compute_cost(h, l) <= max_cost and file_size_mb(b, h) <= max_size_mb
]

print(f"max cost {max_cost} (v11: 10,240) | max file size {max_size_mb} MB")
print("allowed architectures (buckets x hidden x layer2):")
for architecture in architectures:
    b, h, l = parse_architecture(architecture)
    print(f"  {architecture}: cost {compute_cost(h, l)}, file {file_size_mb(b, h):.1f} MB")

## Loading the data: streamed from Lichess, quiet positions with a deep evaluation

In [ ]:
PIECE_CODES = {"P": 1, "N": 2, "B": 3, "R": 4, "Q": 5, "K": 6, "p": 7, "n": 8, "b": 9, "r": 10, "q": 11, "k": 12}
DIGITS = "12345678"


def parse_fen(fen):
    board_part, color = fen.split(" ")[:2]
    row = np.zeros(64, dtype=np.uint8)
    index = 0
    for char in board_part:
        if char == "/":
            continue
        if char in DIGITS:
            index += int(char)
        else:
            row[index ^ 56] = PIECE_CODES[char]
            index += 1
    return row, color == "w"


def square_index(square):
    return (int(square[1]) - 1) * 8 + (ord(square[0]) - ord("a"))


def is_quiet(row, move):
    if len(move) == 5:
        return False
    from_square, to_square = square_index(move[:2]), square_index(move[2:4])
    if row[to_square] != 0:
        return False
    if row[from_square] in (1, 7) and move[0] != move[2]:
        return False
    return True


def score_to_label(cp, mate):
    if mate is not None:
        return 1.0 if mate > 0 else -1.0
    return 2 / (1 + math.exp(-0.00368208 * cp)) - 1


def parse_chunk(chunk):
    rows, sides, values = [], [], []
    num_lines = 0
    for line in chunk.splitlines():
        num_lines += 1
        data = json.loads(line)
        if not data["evals"]:
            continue
        deepest = max(data["evals"], key=lambda e: e["depth"])
        pv = deepest["pvs"][0]
        if deepest["depth"] < min_depth or not pv.get("line"):
            continue
        row, white = parse_fen(data["fen"])
        if not is_quiet(row, pv["line"].split()[0]):
            continue
        rows.append(row[0::2] | (row[1::2] << 4))
        sides.append(white)
        values.append(score_to_label(pv.get("cp"), pv.get("mate")))
    packed = np.array(rows, dtype=np.uint8).reshape(-1, 32)
    return packed, np.array(sides, dtype=np.uint8), np.array(values, dtype=np.float16), num_lines


def read_chunks(reader, chunk_size=16_000_000):
    rest = b""
    while True:
        block = reader.read(chunk_size)
        if not block:
            break
        block = rest + block
        end = block.rfind(b"\n") + 1
        rest = block[end:]
        yield block[:end]
    if rest:
        yield rest


def load_positions(max_positions, max_minutes):
    packed = np.zeros((max_positions, 32), dtype=np.uint8)
    side = np.zeros(max_positions, dtype=np.uint8)
    labels = np.zeros(max_positions, dtype=np.float16)
    count, num_lines, next_log = 0, 0, 5_000_000
    start = time.time()

    with urllib.request.urlopen(url) as response, multiprocessing.get_context("fork").Pool(num_workers) as pool:
        chunks = read_chunks(zstandard.ZstdDecompressor().stream_reader(response))
        pending = collections.deque()
        while count < max_positions and time.time() - start < max_minutes * 60:
            while len(pending) < 2 * num_workers:
                chunk = next(chunks, None)
                if chunk is None:
                    break
                pending.append(pool.apply_async(parse_chunk, (chunk,)))
            if not pending:
                break

            chunk_packed, chunk_side, chunk_labels, chunk_lines = pending.popleft().get()
            size = min(len(chunk_labels), max_positions - count)
            packed[count:count + size] = chunk_packed[:size]
            side[count:count + size] = chunk_side[:size]
            labels[count:count + size] = chunk_labels[:size]
            count += size
            num_lines += chunk_lines
            if count >= next_log:
                log(f"Loaded {count:,} / {max_positions:,} positions from {num_lines:,} lines")
                next_log += 5_000_000

    log(f"Done: {count:,} positions from {num_lines:,} lines in {(time.time() - start) / 60:.0f} min")
    return packed[:count], side[:count], labels[:count]

## King buckets: which weight set is used for each square of the own king

In [ ]:
BUCKET_LAYOUTS = {
    1: [[0, 0, 0, 0]] * 8,
    4: [[0, 0, 1, 1], [2, 2, 2, 2]] + [[3, 3, 3, 3]] * 6,
    8: [[0, 1, 2, 3], [4, 4, 5, 5], [6, 6, 6, 6]] + [[7, 7, 7, 7]] * 5,
    16: [[0, 1, 2, 3], [4, 5, 6, 7], [8, 8, 9, 9], [10, 10, 11, 11]] + [[12, 12, 13, 13]] * 2 + [[14, 14, 15, 15]] * 2,
}


def bucket_table(num_buckets):
    table = []
    for rank in BUCKET_LAYOUTS[num_buckets]:
        table += rank + rank[::-1]
    return table


for num_buckets in bucket_options:
    print(f"{num_buckets} buckets (rank 8 at the top, own king's view):")
    table = bucket_table(num_buckets)
    for rank in range(7, -1, -1):
        print("  " + " ".join(f"{bucket:2d}" for bucket in table[rank * 8:rank * 8 + 8]))

## Features: pieces per king bucket (mirrored), pawn structure, piece counts and mobility

Mobility is measured on the real board and the direction is then turned to the view of the perspective (flipped for black, mirrored when the king is on the e-h files), exactly like the squares of the pieces.

In [ ]:
RAYS = [(1, 0), (-1, 0), (0, 1), (0, -1), (1, 1), (1, -1), (-1, 1), (-1, -1)]
JUMPS = [(2, 1), (2, -1), (-2, 1), (-2, -1), (1, 2), (1, -2), (-1, 2), (-1, -2)]
MOBILITY_DIRECTIONS = [range(8), range(4, 8), range(4), range(8), range(8)]
COUNT_FEATURES = 100
MOBILITY_FEATURES = 640


def orient(directions, direction, flip, mirror):
    rank_step, file_step = directions[direction]
    return directions.index((-rank_step if flip else rank_step, -file_step if mirror else file_step))


PIECE_TYPES = torch.tensor([0, 0, 1, 2, 3, 4, 5, 0, 1, 2, 3, 4, 5], device=device)
PIECE_COLORS = torch.tensor([0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], device=device)
SQUARES = torch.arange(64, device=device)
COUNT_CODES = torch.tensor([1, 2, 3, 4, 5, 7, 8, 9, 10, 11], device=device)
COUNT_OFFSETS = torch.arange(10, device=device) * 10
MAX_ACTIVE = 256

VALID_DIRECTIONS = torch.zeros(6, 8, dtype=torch.bool, device=device)
for kind, directions in enumerate(MOBILITY_DIRECTIONS):
    VALID_DIRECTIONS[kind + 1, list(directions)] = True


def orientation_table(directions):
    return torch.tensor(
        [[[orient(directions, d, flip, mirror) for d in range(8)] for mirror in (0, 1)] for flip in (0, 1)],
        device=device,
    )


RAY_ORIENT = orientation_table(RAYS)
JUMP_ORIENT = orientation_table(JUMPS)


def unpack(data_packed):
    low = data_packed & 15
    high = data_packed >> 4
    return torch.stack([low, high], dim=2).reshape(-1, 64).long()


def adjacent_files(x):
    result = torch.zeros_like(x)
    result[..., 1:] |= x[..., :-1]
    result[..., :-1] |= x[..., 1:]
    return result


def any_above(x):
    result = torch.zeros_like(x)
    for rank in range(6, -1, -1):
        result[:, rank] = result[:, rank + 1] | x[:, rank + 1]
    return result


def any_below(x):
    result = torch.zeros_like(x)
    for rank in range(1, 8):
        result[:, rank] = result[:, rank - 1] | x[:, rank - 1]
    return result


def pawn_structure(board):
    white = (board == 1).view(-1, 8, 8)
    black = (board == 7).view(-1, 8, 8)
    structure = []
    for own, enemy, enemy_ahead in [(white, black, any_above), (black, white, any_below)]:
        passed = own & ~enemy_ahead(enemy | adjacent_files(enemy))
        isolated = own & ~adjacent_files(own.any(dim=1, keepdim=True))
        doubled = own & (own.sum(dim=1, keepdim=True) >= 2)
        structure.append([passed.view(-1, 64), isolated.view(-1, 64), doubled.view(-1, 64)])
    return structure


def shifted(x, rank_step, file_step):
    result = torch.zeros_like(x)
    target_ranks = slice(max(0, -rank_step), min(8, 8 - rank_step))
    target_files = slice(max(0, -file_step), min(8, 8 - file_step))
    source_ranks = slice(max(0, rank_step), min(8, 8 + rank_step))
    source_files = slice(max(0, file_step), min(8, 8 + file_step))
    result[:, target_ranks, target_files] = x[:, source_ranks, source_files]
    return result


def mobility_lengths(board):
    empty = (board == 0).view(-1, 8, 8)
    rays = []
    for rank_step, file_step in RAYS:
        free = torch.ones_like(empty)
        length = torch.zeros(empty.shape, dtype=torch.long, device=board.device)
        for step in range(1, 8):
            free = free & shifted(empty, step * rank_step, step * file_step)
            length += free.long()
        rays.append(length.view(-1, 64))
    jumps = [shifted(empty, rank_step, file_step).long().view(-1, 64) for rank_step, file_step in JUMPS]
    return torch.stack(rays, dim=1), torch.stack(jumps, dim=1)


def count_features(board, perspective, base):
    counts = (board.unsqueeze(1) == COUNT_CODES.view(1, 10, 1)).sum(dim=2).clamp(max=9)
    if perspective == 1:
        counts = counts.roll(5, dims=1)
    return base + COUNT_OFFSETS + counts


def mobility_features(board, lengths, perspective, mirrored, base, padding):
    rays, jumps = lengths
    kinds = PIECE_TYPES[board]
    knights = (kinds == 1).unsqueeze(1)
    length = torch.where(knights, jumps, rays)
    ray_directions = RAY_ORIENT[perspective][mirrored].unsqueeze(2)
    jump_directions = JUMP_ORIENT[perspective][mirrored].unsqueeze(2)
    directions = torch.where(knights, jump_directions, ray_directions)
    enemy = (PIECE_COLORS[board] != perspective).long() * 320
    index = base + (enemy + (kinds - 1) * 64).unsqueeze(1) + directions * 8 + length
    valid = VALID_DIRECTIONS[kinds].transpose(1, 2)
    return index.masked_fill(~valid, padding).view(-1, 512)


def perspective_features(board, structure, lengths, buckets, num_buckets, perspective):
    flip = 56 * perspective
    king = (board == (6 if perspective == 0 else 12)).long().argmax(dim=1) ^ flip
    mirrored = (king % 8 >= 4).long()
    squares = (SQUARES ^ flip) ^ (mirrored * 7).unsqueeze(1)
    count_base = num_buckets * 768 + 384
    mobility_base = count_base + COUNT_FEATURES
    padding = mobility_base + MOBILITY_FEATURES

    own = (PIECE_COLORS[board] != perspective).long() * 384
    pieces = buckets[king].unsqueeze(1) * 768 + own + PIECE_TYPES[board] * 64 + squares
    active = [pieces.masked_fill(board == 0, padding)]

    for kind in range(3):
        own_pawns = structure[perspective][kind]
        enemy_pawns = structure[1 - perspective][kind]
        pawns = num_buckets * 768 + kind * 128 + (~own_pawns).long() * 64 + squares
        active.append(pawns.masked_fill(~(own_pawns | enemy_pawns), padding))

    active.append(count_features(board, perspective, count_base))
    active.append(mobility_features(board, lengths, perspective, mirrored, mobility_base, padding))
    return torch.cat(active, dim=1).sort(dim=1).values[:, :MAX_ACTIVE]


def features(board, white, buckets, num_buckets):
    structure = pawn_structure(board)
    lengths = mobility_lengths(board)
    x_white = perspective_features(board, structure, lengths, buckets, num_buckets, 0)
    x_black = perspective_features(board, structure, lengths, buckets, num_buckets, 1)
    white = white.bool().unsqueeze(1)
    return torch.where(white, x_white, x_black), torch.where(white, x_black, x_white)

## Model

In [ ]:
class NNUE(nn.Module):

    def __init__(self, num_buckets, num_hidden, num_layer2):
        super().__init__()
        self.num_buckets = num_buckets
        self.register_buffer("bucket_table", torch.tensor(bucket_table(num_buckets)))
        width = num_hidden + 1
        bound = 1 / math.sqrt(768)

        self.bucket_weights = nn.Parameter(torch.empty(num_buckets, 768, width).uniform_(-bound, bound))
        self.shared_weights = nn.Parameter(torch.zeros(768, width))
        self.count_weights = nn.Parameter(torch.zeros(2, 6, width))
        self.pawn_weights = nn.Parameter(torch.zeros(384, width))
        self.piece_count_weights = nn.Parameter(torch.zeros(COUNT_FEATURES, width))
        self.mobility_weights = nn.Parameter(torch.zeros(MOBILITY_FEATURES, width))
        self.feature_bias = nn.Parameter(torch.empty(num_hidden).uniform_(-bound, bound))
        with torch.no_grad():
            values = torch.tensor(PIECE_VALUES, dtype=torch.float32) * CP_SCALE
            self.bucket_weights[:, :, -1] = 0
            self.count_weights[0, :, -1] = values
            self.count_weights[1, :, -1] = -values

        if num_layer2 > 0:
            self.hidden = nn.Linear(in_features=2 * num_hidden, out_features=num_layer2)
            self.output = nn.Linear(in_features=num_layer2, out_features=1)
        else:
            self.hidden = None
            self.output = nn.Linear(in_features=2 * num_hidden, out_features=1)

    def feature_table(self):
        counts = self.count_weights.repeat_interleave(64, dim=1).reshape(768, -1)
        pieces = self.bucket_weights + self.shared_weights + counts
        padding = torch.zeros(1, pieces.shape[2], device=pieces.device)
        extra = [self.pawn_weights, self.piece_count_weights, self.mobility_weights, padding]
        return torch.cat([pieces.reshape(-1, pieces.shape[2])] + extra)

    def forward(self, x_us, x_them):
        table = self.feature_table()
        padding = table.shape[0] - 1
        us = nn.functional.embedding_bag(x_us, table, mode="sum", padding_idx=padding)
        them = nn.functional.embedding_bag(x_them, table, mode="sum", padding_idx=padding)
        psqt = (us[:, -1] - them[:, -1]) / 2

        us = torch.clamp(us[:, :-1] + self.feature_bias, 0, 1)
        them = torch.clamp(them[:, :-1] + self.feature_bias, 0, 1)
        x = torch.cat([us, them], dim=1)
        if self.hidden is not None:
            x = torch.clamp(self.hidden(x), 0, 1)
        return self.output(x)[:, 0] + psqt


def build_model(architecture):
    num_buckets, num_hidden, num_layer2 = parse_architecture(architecture)
    return NNUE(num_buckets=num_buckets, num_hidden=num_hidden, num_layer2=num_layer2).to(device)

## Engine file (nnue.bin, format NNU3) and a Python version of the engine evaluation

The Rust engine computes the same inputs; this Python version was checked against the engine on almost 10,000 positions before this notebook was made.

In [ ]:
ENGINE_PARTS = ["feature_weights", "feature_bias", "psqt", "hidden_weight", "hidden_bias", "output_weight", "output_bias"]


def fold_weights(weights):
    bucket_weights = weights["bucket_weights"]
    num_buckets, _, width = bucket_weights.shape
    num_hidden = width - 1
    counts = np.repeat(weights["count_weights"], 64, axis=1).reshape(768, width)
    pieces = bucket_weights + weights["shared_weights"] + counts
    table = np.concatenate([pieces.reshape(-1, width), weights["pawn_weights"], weights["piece_count_weights"], weights["mobility_weights"]])
    num_layer2 = weights["hidden.weight"].shape[0] if "hidden.weight" in weights else 0
    return {
        "num_buckets": num_buckets,
        "mirror": 1,
        "pawn_features": 1,
        "num_hidden": num_hidden,
        "num_layer2": num_layer2,
        "count_features": 1,
        "mobility_features": 1,
        "buckets": weights["bucket_table"],
        "feature_weights": table[:, :num_hidden],
        "feature_bias": weights["feature_bias"],
        "psqt": table[:, num_hidden],
        "hidden_weight": weights["hidden.weight"] if num_layer2 else None,
        "hidden_bias": weights["hidden.bias"] if num_layer2 else None,
        "output_weight": weights["output.weight"],
        "output_bias": weights["output.bias"],
    }


def num_features(net):
    return (
        net["num_buckets"] * 768
        + 384 * net["pawn_features"]
        + COUNT_FEATURES * net["count_features"]
        + MOBILITY_FEATURES * net["mobility_features"]
    )


def write_nnue(net, path):
    with open(path, "wb") as f:
        f.write(b"NNU3")
        f.write(struct.pack(
            "<7I", net["num_buckets"], net["mirror"], net["pawn_features"], net["num_hidden"], net["num_layer2"],
            net["count_features"], net["mobility_features"],
        ))
        f.write(np.asarray(net["buckets"], dtype="<u4").tobytes())
        for name in ENGINE_PARTS:
            if net[name] is not None:
                f.write(np.asarray(net[name], dtype="<f4").tobytes())


def read_nnue(path):
    with open(path, "rb") as f:
        data = f.read()
    assert data[:4] == b"NNU3", "not an NNU3 file"
    names = ["num_buckets", "mirror", "pawn_features", "num_hidden", "num_layer2", "count_features", "mobility_features"]
    net = dict(zip(names, struct.unpack("<7I", data[4:32])))
    num_hidden, num_layer2 = net["num_hidden"], net["num_layer2"]
    shapes = {
        "feature_weights": (num_features(net), num_hidden),
        "feature_bias": (num_hidden,),
        "psqt": (num_features(net),),
        "hidden_weight": (num_layer2, 2 * num_hidden),
        "hidden_bias": (num_layer2,),
        "output_weight": (1, num_layer2 if num_layer2 else 2 * num_hidden),
        "output_bias": (1,),
    }

    net["buckets"] = np.frombuffer(data, dtype="<u4", count=64, offset=32)
    values = np.frombuffer(data, dtype="<f4", offset=32 + 64 * 4)
    start = 0
    for name in ENGINE_PARTS:
        size = math.prod(shapes[name])
        net[name] = values[start:start + size].reshape(shapes[name])
        start += size
    assert start == len(values), "unexpected file size"
    return net


def reference_pawn_structure(row):
    structure = []
    for square in range(64):
        code = row[square]
        if code not in (1, 7):
            continue
        color = 0 if code == 1 else 1
        enemy = 7 if color == 0 else 1
        rank, file = divmod(square, 8)
        ahead = range(rank + 1, 8) if color == 0 else range(rank)
        near = [f for f in (file - 1, file, file + 1) if 0 <= f < 8]
        neighbours = [f for f in (file - 1, file + 1) if 0 <= f < 8]
        if not any(row[r * 8 + f] == enemy for r in ahead for f in near):
            structure.append((color, 0, square))
        if not any(row[r * 8 + f] == code for r in range(8) for f in neighbours):
            structure.append((color, 1, square))
        if any(row[r * 8 + file] == code for r in range(8) if r != rank):
            structure.append((color, 2, square))
    return structure


def reference_counts(row):
    return [(color, kind, int(np.sum(row == 1 + kind + 6 * color))) for color in (0, 1) for kind in range(5)]


def reference_mobility(row):
    mobility = []
    for square in range(64):
        code = int(row[square])
        kind = (code - 1) % 6 - 1
        if code == 0 or kind < 0:
            continue
        color = 0 if code <= 6 else 1
        rank, file = divmod(square, 8)
        directions = JUMPS if kind == 0 else RAYS
        max_steps = 1 if kind == 0 else 7
        for direction in MOBILITY_DIRECTIONS[kind]:
            rank_step, file_step = directions[direction]
            length = 0
            while length < max_steps:
                r, f = rank + rank_step * (length + 1), file + file_step * (length + 1)
                if not (0 <= r < 8 and 0 <= f < 8) or row[r * 8 + f] != 0:
                    break
                length += 1
            mobility.append((color, kind, direction, length))
    return mobility


def reference_features(net, row, perspective):
    flip = 56 * perspective
    king = int(np.argmax(row == (6 if perspective == 0 else 12))) ^ flip
    mirror = 7 if net["mirror"] and king % 8 >= 4 else 0
    bucket = int(net["buckets"][king])
    active = []
    for square in range(64):
        code = int(row[square])
        if code == 0:
            continue
        own = 0 if (code <= 6) == (perspective == 0) else 384
        active.append(bucket * 768 + own + (code - 1) % 6 * 64 + (square ^ flip ^ mirror))

    base = net["num_buckets"] * 768
    if net["pawn_features"]:
        for color, kind, square in reference_pawn_structure(row):
            own = 0 if color == perspective else 64
            active.append(base + kind * 128 + own + (square ^ flip ^ mirror))
        base += 384
    if net["count_features"]:
        for color, kind, count in reference_counts(row):
            own = 0 if color == perspective else 50
            active.append(base + own + kind * 10 + min(count, 9))
        base += COUNT_FEATURES
    if net["mobility_features"]:
        for color, kind, direction, length in reference_mobility(row):
            own = 0 if color == perspective else 320
            directions = JUMPS if kind == 0 else RAYS
            active.append(base + own + kind * 64 + orient(directions, direction, flip, mirror) * 8 + length)
    return active


def reference_eval(net, row, white):
    us, them = (0, 1) if white else (1, 0)
    accumulators, psqt = [], []
    for perspective in (0, 1):
        active = reference_features(net, row, perspective)
        accumulators.append(np.clip(net["feature_bias"] + net["feature_weights"][active].sum(axis=0), 0, 1))
        psqt.append(net["psqt"][active].sum())
    x = np.concatenate([accumulators[us], accumulators[them]])
    if net["num_layer2"] > 0:
        x = np.clip(net["hidden_weight"] @ x + net["hidden_bias"], 0, 1)
    return float((net["output_weight"] @ x + net["output_bias"])[0] + (psqt[us] - psqt[them]) / 2)

In [ ]:
def model_weights(model):
    return {name: value.detach().cpu().numpy() for name, value in model.state_dict().items()}


def export_model(model, path):
    write_nnue(fold_weights(model_weights(model)), path)


def predict(model, row, white):
    board = torch.from_numpy(np.asarray(row)).long().unsqueeze(0).to(device)
    data_white = torch.tensor([white], device=device)
    with torch.no_grad():
        x_us, x_them = features(board, data_white, model.bucket_table, model.num_buckets)
        return model(x_us, x_them).item()

## Training functions

In [ ]:
loss_fn = nn.MSELoss()


def batches(data, indices, model, shuffle):
    data_packed, data_side, data_labels = data
    if shuffle:
        indices = indices[torch.randperm(len(indices), device=indices.device)]
    for start in range(0, len(indices), batch_size):
        batch = indices[start:start + batch_size].long()
        board = unpack(data_packed[batch].to(device, non_blocking=True))
        white = data_side[batch].to(device, non_blocking=True).bool()
        x_us, x_them = features(board, white, model.bucket_table, model.num_buckets)
        labels = data_labels[batch].to(device, non_blocking=True).float()
        targets = torch.where(white, labels, -labels)
        yield x_us, x_them, targets


def train_one_epoch(model, optimizer, data, indices):
    model.train()
    total_loss, num_batches = 0., 0

    for x_us, x_them, targets in batches(data, indices, model, shuffle=True):
        preds = torch.tanh(model(x_us, x_them))
        loss = loss_fn(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.detach()
        num_batches += 1

    return (total_loss / num_batches).item()


def eval_model(model, data, indices):
    model.eval()
    total_loss, num_batches = 0., 0
    total_abs_error, num_preds = 0., 0

    with torch.no_grad():
        for x_us, x_them, targets in batches(data, indices, model, shuffle=False):
            preds = torch.tanh(model(x_us, x_them))

            total_loss += loss_fn(preds, targets)
            num_batches += 1

            total_abs_error += (preds - targets).abs().sum()
            num_preds += targets.shape[0]

    return (total_loss / num_batches).item(), (total_abs_error / num_preds).item()


def measure_speed(architecture, learning_rate, data, indices, num_batches=300, warmup=50):
    model = build_model(architecture)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    model.train()

    for i, (x_us, x_them, targets) in enumerate(batches(data, indices, model, shuffle=True)):
        if i == warmup:
            synchronize()
            start = time.time()
        if i == num_batches:
            break
        loss = loss_fn(torch.tanh(model(x_us, x_them)), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    synchronize()
    return (num_batches - warmup) * batch_size / (time.time() - start)

## Quick check of the whole pipeline

In [ ]:
log("Quick check of the whole pipeline on a small sample...")
sample = tuple(torch.from_numpy(array).to(device) for array in load_positions(20_000, max_minutes=10))
sample_idx = torch.arange(len(sample[2]), device=device, dtype=torch.int32)

check_model = build_model("4x32x8")
check_optimizer = torch.optim.Adam(check_model.parameters(), lr=1e-3)
train_loss = train_one_epoch(check_model, check_optimizer, sample, sample_idx)
val_loss, val_mae = eval_model(check_model, sample, sample_idx)
log(f"Small model: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}")
log(f"Small model speed: {measure_speed('4x32x8', 1e-3, sample, sample_idx, num_batches=4, warmup=1):,.0f} positions/s")

with torch.no_grad():
    for param in check_model.parameters():
        param.add_(0.01 * torch.randn_like(param))
check_path = os.path.join(out_dir, "check.bin")
export_model(check_model, check_path)
check_net = read_nnue(check_path)
os.remove(check_path)

padding = check_model.num_buckets * 768 + 384 + COUNT_FEATURES + MOBILITY_FEATURES
feature_errors, max_error, most_active = 0, 0., 0
for i in range(300):
    row = unpack(sample[0][i:i + 1])[0].cpu().numpy()
    white = bool(sample[1][i])
    board = torch.from_numpy(row).unsqueeze(0).to(device)
    x_us, x_them = features(board, torch.tensor([white], device=device), check_model.bucket_table, check_model.num_buckets)
    us = 0 if white else 1
    for x, perspective in [(x_us, us), (x_them, 1 - us)]:
        most_active = max(most_active, len(reference_features(check_net, row, perspective)))
        if sorted(index for index in x[0].tolist() if index < padding) != sorted(reference_features(check_net, row, perspective)):
            feature_errors += 1
    max_error = max(max_error, abs(predict(check_model, row, white) - reference_eval(check_net, row, white)))

log(f"Feature mismatches: {feature_errors}, most active inputs: {most_active} of {MAX_ACTIVE}, largest difference between the model and the engine file: {max_error:.6f}")
assert feature_errors == 0, "the features differ from the engine"
assert max_error < 1e-3, "the engine file does not match the model"
log("Quick check passed")

## Data

In [ ]:
log(f"Start on {device}, streaming data from Lichess with {num_workers} workers...")
arrays = load_positions(max_positions, max_load_minutes)
num_positions = len(arrays[2])
data_gb = sum(array.nbytes for array in arrays) / 1e9
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if device.type == "cuda" else 0
data_device = device if data_gb < gpu_data_fraction * gpu_gb else torch.device("cpu")
data = tuple(torch.from_numpy(array).to(data_device) for array in arrays)
del arrays
log(f"Data: {num_positions:,} positions, {data_gb:.1f} GB, kept on {data_device}")

perm = torch.randperm(num_positions, generator=torch.Generator().manual_seed(42)).int().to(data_device)
val_idx = perm[:val_size]
test_idx = perm[val_size:val_size + test_size]
train_idx = perm[val_size + test_size:]
tune_train_idx = train_idx[:tune_train_size]
tune_val_idx = val_idx[:tune_val_size]
log(f"Train: {len(train_idx):,}, val: {len(val_idx):,}, test: {len(test_idx):,}")
if device.type == "cuda":
    log(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB of {gpu_gb:.1f} GB")

## Hyperparameter tuning

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    architecture = trial.suggest_categorical("architecture", architectures)
    learning_rate = trial.suggest_float("learning_rate", 3e-4, 3e-3, log=True)

    model = build_model(architecture)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(tune_epochs):
        train_one_epoch(model, optimizer, data, tune_train_idx)
        val_loss, val_mae = eval_model(model, data, tune_val_idx)
        trial.report(val_loss, epoch)
        if trial.should_prune():
            log(f"Trial {trial.number}: stopped early after epoch {epoch + 1} (val loss {val_loss:.4f}, {architecture}, lr {learning_rate:.5f})")
            raise optuna.TrialPruned()

    log(f"Trial {trial.number}: val loss {val_loss:.4f}, val MAE {val_mae:.4f}, {architecture}, lr {learning_rate:.5f}")
    return val_loss


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)
study.optimize(objective, n_trials=num_trials, timeout=tune_hours * 3600)

best_params = study.best_params
log(f"Best params: {best_params} (val loss {study.best_value:.4f})")
print(study.trials_dataframe(attrs=("number", "value", "params", "state")).sort_values("value"))

## Final training

In [ ]:
num_buckets, num_hidden, num_layer2 = parse_architecture(best_params["architecture"])
learning_rate = best_params["learning_rate"]
deadline = notebook_start + total_hours * 3600 - reserve_minutes * 60

speed = measure_speed(best_params["architecture"], learning_rate, data, train_idx)
epoch_seconds = 1.1 * len(train_idx) / speed
num_epochs = max(1, min(max_epochs, int((deadline - time.time()) / epoch_seconds)))
log(f"Speed: {speed:,.0f} positions/s, about {epoch_seconds / 60:.0f} min per epoch, so {num_epochs} epochs fit in the time budget")

model = build_model(best_params["architecture"])
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=learning_rate / 100)
weights_path = os.path.join(out_dir, "nnue_weights.npz")
engine_path = os.path.join(out_dir, "nnue.bin")
print(model)

history = {"train_loss": [], "val_loss": [], "val_mae": [], "learning_rate": []}
best_val_loss = float("inf")
best_state = None

for epoch in range(num_epochs):
    if epoch > 0 and time.time() + epoch_seconds > deadline:
        log("Stopping early: not enough time left for another epoch")
        break

    current_lr = optimizer.param_groups[0]["lr"]
    train_loss = train_one_epoch(model, optimizer, data, train_idx)
    val_loss, val_mae = eval_model(model, data, val_idx)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_mae"].append(val_mae)
    history["learning_rate"].append(current_lr)
    log(f"Epoch {epoch + 1}/{num_epochs}: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}, lr {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {name: value.detach().clone() for name, value in model.state_dict().items()}
        np.savez(weights_path, **model_weights(model))
        export_model(model, engine_path)
        log("Checkpoint saved: nnue_weights.npz and nnue.bin")

## Test

In [ ]:
model.load_state_dict(best_state)
test_loss, test_mae = eval_model(model, data, test_idx)
log(f"Test: loss {test_loss:.4f}, MAE {test_mae:.4f}")

## Sanity check and piece values

The piece values are measured like this: the evaluation of a position where black misses one piece, minus the evaluation of the same position with all pieces. For comparison, v11 gave: opening pawn +64, knight +363, bishop +417, rook +446, queen +667; endgame pawn +165, 2 pawns +368, rook +923.

In [ ]:
test_fens = {
    "start position": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "white a bishop down, white to move": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQK1NR w KQkq - 0 1",
    "white a queen down, white to move": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNB1KBNR w KQkq - 0 1",
    "rook against 3 pawns, white to move (about +200)": "1nbqkbnr/pppppppp/8/8/8/8/3PPPPP/RNBQKBNR w KQk - 0 1",
    "lost game 1, black a bishop down (Stockfish -639)": "2r3k1/5p1p/bq2p1p1/pN1pP3/P2P2n1/RP1Q1NP1/5PBP/6K1 b - - 0 22",
    "lost game 2, black to move (Stockfish -387)": "2krr3/1pp2p1p/p1nb3q/3p4/Q1PP2pN/3bP1P1/P2N1PBP/R2R2K1 b - - 0 16",
}

engine_net = read_nnue(engine_path)
sanity = {}
for name, fen in test_fens.items():
    row, white = parse_fen(fen)
    raw = predict(model, row, white)
    engine_raw = reference_eval(engine_net, row, white)
    sanity[name] = {"fen": fen, "raw": raw}
    log(f"{name}: {raw / CP_SCALE:+.0f} cp for the side to move (engine file: {engine_raw / CP_SCALE:+.0f} cp)")


material_tests = {
    "opening": ("rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1", {
        "pawn": "rnbqkbnr/1ppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "knight": "r1bqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "bishop": "rn1qkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "rook": "1nbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQk - 0 1",
        "queen": "rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    }),
    "endgame": ("r3k3/ppp2ppp/8/8/8/8/PPP2PPP/R3K3 w - - 0 1", {
        "pawn": "r3k3/ppp2pp1/8/8/8/8/PPP2PPP/R3K3 w - - 0 1",
        "2 pawns": "r3k3/ppp2p2/8/8/8/8/PPP2PPP/R3K3 w - - 0 1",
        "rook": "4k3/ppp2ppp/8/8/8/8/PPP2PPP/R3K3 w - - 0 1",
    }),
}

piece_values = {}
for phase, (start_fen, missing) in material_tests.items():
    start_value = predict(model, *parse_fen(start_fen))
    for name, fen in missing.items():
        value = (predict(model, *parse_fen(fen)) - start_value) / CP_SCALE
        piece_values[f"{phase}: black without {name}"] = value
        log(f"{phase}, black without {name}: {value:+.0f} cp for white")

params = {
    "architecture": best_params["architecture"],
    "num_buckets": num_buckets,
    "num_hidden": num_hidden,
    "num_layer2": num_layer2,
    "learning_rate": learning_rate,
    "cost": compute_cost(num_hidden, num_layer2),
    "max_cost": max_cost,
    "file_size_mb": file_size_mb(num_buckets, num_hidden),
    "num_positions": num_positions,
    "min_depth": min_depth,
    "num_epochs": len(history["train_loss"]),
    "positions_per_second": speed,
}
with open(os.path.join(out_dir, "nnue_mobility.json"), "w", encoding="utf-8") as f:
    json.dump({"params": params, "history": history, "test_loss": test_loss, "test_mae": test_mae, "sanity": sanity, "piece_values": piece_values}, f, indent=2)
log("Saved nnue_mobility.json")

## Plots

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs, history["val_mae"], label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs, history["learning_rate"], label="Learning rate")
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.yscale("log")
plt.legend()

plt.savefig(os.path.join(out_dir, "nnue_mobility_curves.png"), bbox_inches="tight")
plt.show()

In [ ]:
try:
    optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.savefig(os.path.join(out_dir, "tuning_history.png"), bbox_inches="tight")
    plt.show()
    optuna.visualization.matplotlib.plot_param_importances(study)
    plt.savefig(os.path.join(out_dir, "tuning_importances.png"), bbox_inches="tight")
    plt.show()
except Exception as error:
    log(f"Could not make tuning plots: {error}")

study.trials_dataframe(attrs=("number", "value", "params", "state")).to_csv(os.path.join(out_dir, "tuning_trials.csv"), index=False)
log(f"Done. Download nnue.bin for the engine. Files: {sorted(os.listdir(out_dir))}")